In [1]:
from pathlib import Path
from urllib.request import urlopen

REPO = "contrivancecompanychicago/imhotep-bizarro-secretary-backend"
COMMIT = "0630117f4228f82ba39c2ae4ea5dacd9dcaadba0"

research_dir = Path("/content/imhotep-research")
research_dir.mkdir(parents=True, exist_ok=True)

for filename in ["README.md", "baseline.py"]:
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/research/{filename}"
    with urlopen(url, timeout=30) as response:
        (research_dir / filename).write_bytes(response.read())
    print(f"Loaded: {filename}")

print("\nSETUP COMPLETE")
print("Repository commit:", COMMIT)

Loaded: README.md
Loaded: baseline.py

SETUP COMPLETE
Repository commit: 0630117f4228f82ba39c2ae4ea5dacd9dcaadba0


In [2]:
%run /content/imhotep-research/baseline.py

{
  "label": "synthetic smoke demo only; not a study result",
  "method": "token_overlap_v1",
  "query_count": 2,
  "top1_accuracy": 1.0,
  "rankings": [
    {
      "query_id": "Q1",
      "ranked_ids": [
        "SYN-001-N1",
        "SYN-002-N1"
      ],
      "relevant_id": "SYN-001-N1"
    },
    {
      "query_id": "Q2",
      "ranked_ids": [
        "SYN-003-N1"
      ],
      "relevant_id": "SYN-003-N1"
    }
  ]
}


In [3]:
import runpy

baseline = runpy.run_path("/content/imhotep-research/baseline.py")
rank = baseline["rank"]

documents = [
    {
        "id": "SYN-001-N1",
        "text": "Fictional case SYN-001 requests an afternoon appointment."
    },
    {
        "id": "SYN-002-N1",
        "text": "Fictional case SYN-002 requests a morning appointment."
    }
]

query = "visit after noon"

print("Query:", query)
print("Expected document: SYN-001-N1")
print("Retrieved documents:", rank(query, documents))

Query: visit after noon
Expected document: SYN-001-N1
Retrieved documents: []


In [4]:
queries = [
    ("Exact wording", "afternoon appointment"),
    ("Different wording", "visit after noon"),
]

expected = "SYN-001-N1"

for label, query in queries:
    retrieved = rank(query, documents)
    correct = bool(retrieved) and retrieved[0] == expected

    print(f"{label}: {query}")
    print(f"Retrieved: {retrieved}")
    print(f"Correct first result: {correct}\n")

Exact wording: afternoon appointment
Retrieved: ['SYN-001-N1', 'SYN-002-N1']
Correct first result: True

Different wording: visit after noon
Retrieved: []
Correct first result: False



In [5]:
"""
### Development observation 001: Query wording

Method: token_overlap_v1

Documents: Two fictional appointment records, held fixed.

Results:
- "afternoon appointment" retrieved the correct document first.
- "visit after noon" returned no documents.
- Top-1 accuracy across these two development queries: 1/2 = 50%.

Interpretation:
The token-overlap baseline missed a relevant document when the query
used different words with a similar intended meaning.

Limitation:
These are two hand-authored development examples, not a held-out
evaluation or evidence of clinical effectiveness.

Next experiment:
Compare the unchanged baseline with a clearly labeled query-expansion
method on these development examples before testing unseen cases.
"""

'\n### Development observation 001: Query wording\n\nMethod: token_overlap_v1\n\nDocuments: Two fictional appointment records, held fixed.\n\nResults:\n- "afternoon appointment" retrieved the correct document first.\n- "visit after noon" returned no documents.\n- Top-1 accuracy across these two development queries: 1/2 = 50%.\n\nInterpretation:\nThe token-overlap baseline missed a relevant document when the query\nused different words with a similar intended meaning.\n\nLimitation:\nThese are two hand-authored development examples, not a held-out\nevaluation or evidence of clinical effectiveness.\n\nNext experiment:\nCompare the unchanged baseline with a clearly labeled query-expansion\nmethod on these development examples before testing unseen cases.\n'

In [6]:
import re

def expand_query(query):
    expanded = query.lower()
    expanded = re.sub(r"\bafter noon\b", "afternoon", expanded)
    expanded = re.sub(r"\bvisit\b", "appointment", expanded)
    return expanded

methods = {
    "Original baseline": lambda query: query,
    "Rule-based expansion v1": expand_query,
}

for method_name, transform in methods.items():
    correct_count = 0
    print(f"\n{method_name}")

    for label, query in queries:
        search_query = transform(query)
        retrieved = rank(search_query, documents)
        correct = bool(retrieved) and retrieved[0] == expected
        correct_count += int(correct)

        print(f"  Original: {query}")
        print(f"  Searched: {search_query}")
        print(f"  Correct first result: {correct}")

    print(f"  Top-1 accuracy: {correct_count}/{len(queries)}")


Original baseline
  Original: afternoon appointment
  Searched: afternoon appointment
  Correct first result: True
  Original: visit after noon
  Searched: visit after noon
  Correct first result: False
  Top-1 accuracy: 1/2

Rule-based expansion v1
  Original: afternoon appointment
  Searched: afternoon appointment
  Correct first result: True
  Original: visit after noon
  Searched: appointment afternoon
  Correct first result: True
  Top-1 accuracy: 2/2


In [7]:
# The morning record is relevant to "before noon".
# This checks paraphrase retrieval and ranking.
negative_query = "visit before noon"

for method_name, transform in methods.items():
    searched = transform(negative_query)
    retrieved = rank(searched, documents)

    print(f"\n{method_name}")
    print("Query:", negative_query)
    print("Searched:", searched)
    print("Retrieved:", retrieved)


Original baseline
Query: visit before noon
Searched: visit before noon
Retrieved: []

Rule-based expansion v1
Query: visit before noon
Searched: appointment before noon
Retrieved: ['SYN-001-N1', 'SYN-002-N1']


In [8]:
missing_query = "evening appointment"

for method_name, transform in methods.items():
    retrieved = rank(transform(missing_query), documents)
    print(f"{method_name}: {retrieved}")

Original baseline: ['SYN-001-N1', 'SYN-002-N1']
Rule-based expansion v1: ['SYN-001-N1', 'SYN-002-N1']


In [9]:
def rank_with_time_filter(query, documents):
    expanded = expand_query(query)
    expanded = re.sub(r"\bbefore noon\b", "morning", expanded)

    time_terms = {"morning", "afternoon", "evening"}
    required_times = set(expanded.split()) & time_terms

    eligible = [
        doc for doc in documents
        if required_times.issubset(set(doc["text"].lower().split()))
    ]
    return rank(expanded, eligible)

for query in [
    "afternoon appointment",
    "visit after noon",
    "visit before noon",
    "evening appointment",
]:
    print(query, "→", rank_with_time_filter(query, documents))

afternoon appointment → ['SYN-001-N1']
visit after noon → ['SYN-001-N1']
visit before noon → ['SYN-002-N1']
evening appointment → []


In [10]:
development_documents = [
    {
        "id": "DEV-001-N1",
        "case_id": "DEV-001",
        "text": "Fictional case DEV-001 requests an afternoon appointment."
    },
    {
        "id": "DEV-002-N1",
        "case_id": "DEV-002",
        "text": "Fictional case DEV-002 requests a morning appointment."
    },
    {
        "id": "DEV-003-N1",
        "case_id": "DEV-003",
        "text": "Fictional case DEV-003 needs interpreter assistance."
    },
    {
        "id": "DEV-004-N1",
        "case_id": "DEV-004",
        "text": "Fictional case DEV-004 requests transportation assistance."
    },
    {
        "id": "DEV-005-N1",
        "case_id": "DEV-005",
        "text": "Fictional case DEV-005 requests a telephone callback."
    },
]

assert len({doc["id"] for doc in development_documents}) == 5
assert len({doc["case_id"] for doc in development_documents}) == 5

for doc in development_documents:
    print(doc["id"], "→", doc["text"])

print("\nDEVELOPMENT CASES READY:", len(development_documents))

DEV-001-N1 → Fictional case DEV-001 requests an afternoon appointment.
DEV-002-N1 → Fictional case DEV-002 requests a morning appointment.
DEV-003-N1 → Fictional case DEV-003 needs interpreter assistance.
DEV-004-N1 → Fictional case DEV-004 requests transportation assistance.
DEV-005-N1 → Fictional case DEV-005 requests a telephone callback.

DEVELOPMENT CASES READY: 5


In [11]:
development_questions = [
    {
        "query_id": "DQ-001",
        "query": "afternoon appointment",
        "relevant_ids": ["DEV-001-N1"],
    },
    {
        "query_id": "DQ-002",
        "query": "visit before noon",
        "relevant_ids": ["DEV-002-N1"],
    },
    {
        "query_id": "DQ-003",
        "query": "interpreter assistance",
        "relevant_ids": ["DEV-003-N1"],
    },
    {
        "query_id": "DQ-004",
        "query": "transportation assistance",
        "relevant_ids": ["DEV-004-N1"],
    },
    {
        "query_id": "DQ-005",
        "query": "telephone callback",
        "relevant_ids": ["DEV-005-N1"],
    },
    {
        "query_id": "DQ-006",
        "query": "evening appointment",
        "relevant_ids": [],  # No supporting document
    },
]

document_ids = {doc["id"] for doc in development_documents}

assert len({q["query_id"] for q in development_questions}) == 6
assert all(
    set(q["relevant_ids"]).issubset(document_ids)
    for q in development_questions
)

for question in development_questions:
    print(
        question["query_id"],
        "→", question["query"],
        "→", question["relevant_ids"] or "NO MATCH",
    )

print("\nANSWER KEY READY: 5 answerable questions + 1 no-match question")

DQ-001 → afternoon appointment → ['DEV-001-N1']
DQ-002 → visit before noon → ['DEV-002-N1']
DQ-003 → interpreter assistance → ['DEV-003-N1']
DQ-004 → transportation assistance → ['DEV-004-N1']
DQ-005 → telephone callback → ['DEV-005-N1']
DQ-006 → evening appointment → NO MATCH

ANSWER KEY READY: 5 answerable questions + 1 no-match question


In [12]:
retrieval_methods = {
    "Original baseline": rank,
    "Expansion v1": lambda q, docs: rank(expand_query(q), docs),
    "Expansion + time filter": rank_with_time_filter,
}

development_results = []

for method_name, retrieve in retrieval_methods.items():
    correct_top1 = 0
    correct_abstentions = 0
    answerable_count = 0
    no_match_count = 0

    print(f"\n{method_name}")

    for question in development_questions:
        retrieved = retrieve(
            question["query"], development_documents
        )
        relevant = question["relevant_ids"]

        if relevant:
            answerable_count += 1
            correct = bool(retrieved) and retrieved[0] in relevant
            correct_top1 += int(correct)
        else:
            no_match_count += 1
            correct = not retrieved
            correct_abstentions += int(correct)

        development_results.append({
            "method": method_name,
            "query_id": question["query_id"],
            "query": question["query"],
            "relevant_ids": relevant,
            "retrieved_ids": retrieved,
            "correct": correct,
        })

        print(
            question["query_id"],
            "→", retrieved,
            "→", "PASS" if correct else "FAIL"
        )

    print(f"Top-1 correct: {correct_top1}/{answerable_count}")
    print(
        f"Correct abstentions: "
        f"{correct_abstentions}/{no_match_count}"
    )


Original baseline
DQ-001 → ['DEV-001-N1', 'DEV-002-N1'] → PASS
DQ-002 → [] → FAIL
DQ-003 → ['DEV-003-N1', 'DEV-004-N1'] → PASS
DQ-004 → ['DEV-004-N1', 'DEV-003-N1'] → PASS
DQ-005 → ['DEV-005-N1'] → PASS
DQ-006 → ['DEV-001-N1', 'DEV-002-N1'] → FAIL
Top-1 correct: 4/5
Correct abstentions: 0/1

Expansion v1
DQ-001 → ['DEV-001-N1', 'DEV-002-N1'] → PASS
DQ-002 → ['DEV-001-N1', 'DEV-002-N1'] → FAIL
DQ-003 → ['DEV-003-N1', 'DEV-004-N1'] → PASS
DQ-004 → ['DEV-004-N1', 'DEV-003-N1'] → PASS
DQ-005 → ['DEV-005-N1'] → PASS
DQ-006 → ['DEV-001-N1', 'DEV-002-N1'] → FAIL
Top-1 correct: 4/5
Correct abstentions: 0/1

Expansion + time filter
DQ-001 → ['DEV-001-N1'] → PASS
DQ-002 → ['DEV-002-N1'] → PASS
DQ-003 → ['DEV-003-N1', 'DEV-004-N1'] → PASS
DQ-004 → ['DEV-004-N1', 'DEV-003-N1'] → PASS
DQ-005 → ['DEV-005-N1'] → PASS
DQ-006 → [] → PASS
Top-1 correct: 5/5
Correct abstentions: 1/1


In [13]:
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

dataset = {
    "documents": development_documents,
    "questions": development_questions,
}

dataset_bytes = json.dumps(
    dataset, sort_keys=True, ensure_ascii=False
).encode("utf-8")

record = {
    "experiment_id": "IMHOTEP-DEV-001",
    "stage": "development",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "baseline_source_commit": COMMIT,
    "dataset_sha256": hashlib.sha256(dataset_bytes).hexdigest(),
    "limitation": "Hand-authored development examples; not held-out evaluation.",
    "dataset": dataset,
    "results": development_results,
}

output_path = Path("/content/IMHOTEP_DEV_001.json")
output_path.write_text(
    json.dumps(record, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved:", output_path.name)
print("Result rows:", len(development_results))
print("Dataset SHA-256:", record["dataset_sha256"])

files.download(str(output_path))

Saved: IMHOTEP_DEV_001.json
Result rows: 18
Dataset SHA-256: ef4b6af4b4c6034a6a82eaa8d6fcff28e57f017aa927dce0c967b335775cab3c


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>